# Notebook 03 — Geometry Import and Mesh Quality

## What you will learn
1. How to import CDB files from nTopology into MAPDL
2. Critical nTopology CDB import pitfalls (SOLID185 vs SHELL181, NROTAT)
3. What mesh quality metrics mean and why they matter
4. The Jacobian determinant — what it is and why negative = fatal
5. How to interpret aspect ratio, warping, and skewness
6. How to choose the right ANSYS element type for your physics

---

## Part 1: Geometry Import Workflow

### CDB (from nTopology)

nTopology exports meshes in the ANSYS CDB (CDWRITE) format.  The CDB is a
complete database dump containing:
- Node coordinates and IDs
- Element connectivity
- Material definitions
- Section data (for shells)
- Boundary conditions (sometimes — check before importing)

**Critical issues with nTopology CDB exports:**

| Issue | Symptom | Fix |
|-------|---------|-----|
| SOLID185 for thin shells | Shear locking — too stiff in bending | Reassign to SHELL181 |
| Node CS rotations | Wrong DOF directions after CDREAD | `NROTAT,ALL` |
| Missing real constants | Shell section thickness not imported | Define manually |
| Old nTop versions | ETYPE not set correctly | Check ETLIST before solving |

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from ams.geometry.element_selector import print_element_guide, choose_element, ELEMENT_LIBRARY

# ── Print the element reference table ─────────────────────────────────────
print_element_guide()

In [ ]:
# ── Interactive element recommendation ────────────────────────────────────
# Example: choose for an origami shell structure

rec = choose_element(
    spatial_dim       = 3,
    is_thin_shell     = True,
    large_deformation = True,
    high_accuracy     = False,
)
info = ELEMENT_LIBRARY[rec.name]

print(f'Recommended element: {rec.name}')
print(f'Best for:')
for b in info.best_for:
    print(f'  - {b}')
print(f'Avoid for:')
for a in info.avoid_for:
    print(f'  - {a}')
print(f'\nKEYOPT notes:')
for k, v in info.keyopt_notes.items():
    print(f'  KEYOPT({k}): {v}')

---

## Part 2: Mesh Quality Metrics

### Why mesh quality matters

The finite element solution converges to the true solution as the mesh is
refined — but only if the elements are **not too distorted**.

Distorted elements cause:
1. **Ill-conditioning** of the stiffness matrix (large condition number → round-off errors)
2. **Inaccurate integration** — Gauss quadrature assumes the Jacobian is smooth
3. **Solver divergence** — negative Jacobian causes the solve to immediately fail

### Jacobian determinant

For an element with nodes at positions $\mathbf{x}_i$, the Jacobian matrix $\mathbf{J}$
maps from the reference element $[{-1, 1}]^3$ to physical space:

$$J_{ij} = \frac{\partial x_i}{\partial \xi_j} = \sum_k \frac{\partial N_k}{\partial \xi_j} x_{ki}$$

The determinant $\det(\mathbf{J})$ is the local volume ratio:
- $\det(\mathbf{J}) > 0$: element is correctly oriented
- $\det(\mathbf{J}) = 0$: degenerate element (collapsed to a plane or line)
- $\det(\mathbf{J}) < 0$: **inverted element — solver WILL crash**

An inverted element means two nodes have been swapped during mesh generation.
This usually comes from STL imports with non-manifold geometry, or very
aggressive mesh refinement near sharp features.

### Aspect ratio

$$AR = \frac{\ell_{\max}}{\ell_{\min}}$$

where $\ell_{\max}$ and $\ell_{\min}$ are the longest and shortest edges of the element.

- $AR = 1$: perfect equilateral element
- $AR < 10$: acceptable for most problems
- $AR > 20$: likely to cause ill-conditioning
- $AR > 100$: degenerate — usually indicates a meshing error

High AR elements are sometimes intentional (boundary layers in CFD, thin plate
elements modeled with solid elements) — but they require careful handling.

In [ ]:
# ── Visualize element quality metrics ─────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.flatten()

element_examples = [
    {
        'name': 'Perfect square\nAR=1.0, J>0',
        'nodes': np.array([[0,0],[1,0],[1,1],[0,1]]),
        'quality': 'good', 'color': 'lightgreen',
    },
    {
        'name': 'Elongated\nAR=5.0',
        'nodes': np.array([[0,0],[5,0],[5,1],[0,1]]),
        'quality': 'warn', 'color': 'lightyellow',
    },
    {
        'name': 'Skewed\nHigh skewness',
        'nodes': np.array([[0,0],[1,0],[1.5,1],[0.5,1]]),
        'quality': 'warn', 'color': 'lightyellow',
    },
    {
        'name': 'Inverted (J<0)\nFATAL — solver crash',
        'nodes': np.array([[0,0],[1,0],[0,1],[1,1]]),  # swapped top corners
        'quality': 'bad', 'color': 'salmon',
    },
    {
        'name': 'Extremely elongated\nAR=20+',
        'nodes': np.array([[0,0],[10,0],[10,0.5],[0,0.5]]),
        'quality': 'bad', 'color': 'salmon',
    },
    {
        'name': 'Good shell element\nLow warping',
        'nodes': np.array([[0,0],[1,0.05],[1.05,1.05],[0.05,1]]),
        'quality': 'good', 'color': 'lightblue',
    },
]

for ax, elem in zip(axes, element_examples):
    nodes = elem['nodes']
    # Close the polygon
    poly = np.vstack([nodes, nodes[0]])
    ax.fill(poly[:,0], poly[:,1], alpha=0.5, color=elem['color'], edgecolor='navy', linewidth=2)
    for i, (x, y) in enumerate(nodes):
        ax.plot(x, y, 'ko', markersize=6)
        ax.text(x, y, f' {i}', fontsize=9, color='black')
    ax.set_title(elem['name'], fontsize=9)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.2); ax.axis('off')

plt.suptitle('Mesh Quality Examples', fontsize=11)
plt.tight_layout(); plt.show()

print('Green  = good quality  (AR < 5, J > 0.2)')
print('Yellow = marginal      (AR 5-20, J > 0)')
print('Red    = problematic   (AR > 20, or J ≤ 0)')

In [ ]:
# ── The MeshQualityChecker in action (mock data) ──────────────────────────
# In a real run, pass a live mapdl instance:
# from ams.geometry.mesh_quality import MeshQualityChecker
# checker = MeshQualityChecker(mapdl)
# report  = checker.check(raise_on_fail=True)

# Here we simulate a QualityReport for demonstration:
from ams.geometry.mesh_quality import QualityReport

mock_report = QualityReport(
    passed      = True,
    n_nodes     = 1250,
    n_elements  = 1000,
    checks      = {
        'aspect_ratio': {'status': 'PASS', 'value': 4.2, 'msg': 'max=4.20 mean=1.85 (limit 20) 0 bad elements'},
        'jacobian':     {'status': 'PASS', 'value': 0.72, 'msg': 'min=0.7200 (< 0 = inverted element — solve blocked)'},
        'warping_deg':  {'status': 'PASS', 'value': 2.1, 'msg': 'max=2.1° (shell limit 30°)'},
        'manifold':     {'status': 'PASS', 'msg': 'OK — mesh is manifold'},
        'watertight':   {'status': 'WARN', 'value': 200, 'msg': '200 free boundary faces (OK for shells; bad for solids)'},
    }
)
mock_report.print_summary()

---

## Summary

| Metric | Good | Warning | Fatal |
|--------|------|---------|-------|
| Aspect ratio | < 5 | 5–20 | > 20 |
| Jacobian | > 0.5 | 0–0.5 | ≤ 0 |
| Warping (shells) | < 10° | 10–30° | > 30° |
| Manifold | Yes | — | No (geometry error) |

**Always run mesh quality check before solving.** The `MeshQualityChecker` is
gate-checked before `run_solution()` — it will raise RuntimeError if critical
thresholds are violated.

**Next:** `04_boundary_conditions.ipynb`

---

## Part 3: nTopology 5.x — Importing via Abaqus .inp

nTopology 5.x's **Export FE Mesh** block writes Abaqus `.inp` format — not `.cdb`.
The toolbox converts it transparently via `meshio` before MAPDL reads it.

### Why `.inp` and not `.cdb`?

nTopology 5.x removed native CDB export. The supported FEA formats are `.inp` (Abaqus),
`.bdf`/`.nas` (Nastran), and `.k` (LS-DYNA). Abaqus `.inp` is preferred because `meshio`
has a robust reader and a reliable ANSYS CDB writer.

### Data flow

```
nTopology Export FE Mesh
  └─ writes waterbomb_mesh.inp  (Abaqus: *NODE, *ELEMENT, *SURFACE)
         │
         ▼  meshio.read()
  meshio Mesh object  (points array, cells list)
         │
         ▼  meshio.write(format="ansys")
  waterbomb_mesh.cdb  (ANSYS NBLOCK + EBLOCK)
         │
         ▼  mapdl.cdread("DB", ...)  +  NROTAT ALL
  MAPDL mesh database
         │
         ▼  reassign_element_type("SHELL181")
  SHELL181 ready for large-deformation solve
```

meshio maps the nTopology triangle3 shell cells to MAPDL element type 1, which is
immediately upgraded to SHELL181 by `reassign_element_type()`.

### Actual mesh file for this notebook

```
E:/Projects/ansys_sim_toolbox/simulation_meshes/waterbomb_mesh.inp
```
This was exported manually from the Waterbomb.ntop GUI by computing the Export FE Mesh block.

In [ ]:
import sys
sys.path.insert(0, '..')

INP_PATH = "E:/Projects/ansys_sim_toolbox/simulation_meshes/waterbomb_mesh.inp"

# ── Inspect the .inp file with meshio — no MAPDL license required ─────────
import meshio
import numpy as np

mesh = meshio.read(INP_PATH)
pts  = mesh.points   # (N, 3) array, units = mm as exported from nTop

print(f"Nodes  : {len(pts):,}")
print(f"X range: [{pts[:,0].min():.2f}, {pts[:,0].max():.2f}] mm")
print(f"Y range: [{pts[:,1].min():.2f}, {pts[:,1].max():.2f}] mm")
print(f"Z range: [{pts[:,2].min():.4f}, {pts[:,2].max():.4f}] mm  ← negative = trough depth")
print()
print("Cell blocks exported by nTopology:")
for cb in mesh.cells:
    print(f"  type={cb.type:12s}  count={len(cb.data):6d}")

In [ ]:
# ── Identify crease nodes from geometry — the same logic origami_bcs.py uses ─
# Units from nTop: mm.  The BC code converts to metres internally.

Lx = pts[:, 0].max() - pts[:, 0].min()
Ly = pts[:, 1].max() - pts[:, 1].min()
cx = pts[:, 0].min() + Lx / 2   # plate centre X
cy = pts[:, 1].min() + Ly / 2   # plate centre Y

print(f"Plate size : {Lx:.1f} × {Ly:.1f} mm")
print(f"Centre     : ({cx:.2f}, {cy:.2f}) mm")
print()

# crease_tol_mm must be ≥ Mesh Tolerance / 2  (nTop Mesh Tolerance = 1 mm → use 0.6)
tol = 0.6   # mm

x, y = pts[:, 0], pts[:, 1]

crease_masks = {
    "vertical  (x = cx)":        np.abs(x - cx) < tol,
    "horizontal (y = cy)":       np.abs(y - cy) < tol,
    "diag +45° (y−cy = x−cx)":  np.abs((y - cy) - (x - cx)) < tol,
    "diag −45° (y−cy = −(x−cx))": np.abs((y - cy) + (x - cx)) < tol,
}

print(f"Crease node counts  (tol = {tol} mm):")
for name, mask in crease_masks.items():
    print(f"  {name:38s} → {mask.sum():5d} nodes")

# The centre patch (pinned UX=UY=UZ=0)
pin_r = 2.0   # mm — pin_center_radius_mm from YAML
r2    = (x - cx)**2 + (y - cy)**2
centre_mask = r2 <= pin_r**2
print(f"\n  {'centre patch (r < 2 mm)':38s} → {centre_mask.sum():5d} nodes  (these get UX=UY=UZ=0)")

# The 4 outer corners (driven UZ displacement)
corner_tol = Lx * 0.05   # 5% of plate length
corners    = [(cx - Lx/2, cy - Ly/2), (cx + Lx/2, cy - Ly/2),
              (cx - Lx/2, cy + Ly/2), (cx + Lx/2, cy + Ly/2)]
corner_mask = np.zeros(len(pts), dtype=bool)
for (xc, yc) in corners:
    r2c = (x - xc)**2 + (y - yc)**2
    corner_mask |= r2c <= corner_tol**2
print(f"  {'outer corners':38s} → {corner_mask.sum():5d} nodes  (these get UZ = fold_uz_mm)")

### Importing into MAPDL (requires ANSYS license)

```python
import yaml
from ams.resources.manager import kill_ansys_zombies
from ams.mapdl.runner import MAPDLRunner
from ams.geometry.importer import GeometryImporter
from ams.geometry.mesh_quality import MeshQualityChecker

kill_ansys_zombies()

with open('../examples/origami_folding/origami_folding.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

runner = MAPDLRunner(cfg)
mapdl  = runner.connect()
try:
    mapdl.clear()
    gi = GeometryImporter(mapdl)

    # Converts .inp → .cdb via meshio, runs CDREAD, runs NROTAT ALL,
    # then upgrades all elements from SHELL63 (meshio default) to SHELL181
    gi.from_inp(
        "E:/Projects/ansys_sim_toolbox/simulation_meshes/waterbomb_mesh.inp",
        reassign_et="SHELL181",
    )

    checker = MeshQualityChecker(mapdl)
    report  = checker.check(raise_on_fail=True)
    report.print_summary()
finally:
    runner.disconnect()
```

### Generalizing to other nTopology geometries

| What to change | Where |
|---|---|
| Input Block names in `ntop.parameters` | Match your `.ntop` workflow tree |
| `elements.type` and `section_thickness_m` | Match your physical model |
| `bcs.*` — crease coordinates or named components | Match your geometry |
| `expected_outputs: ["*.inp"]` | Always the same for nTop 5.x |

The `NTopDriver` and `from_inp()` are fully geometry-agnostic.
Only the YAML parameters and BC coordinates are geometry-specific.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.tri as mtri

# ── Visualize mesh, crease nodes, and BC regions ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

triangles = []
for cb in mesh.cells:
    if cb.type in ("triangle", "triangle3", "S3", "S3R"):
        triangles.extend(cb.data)
tri_arr = np.array(triangles) if triangles else None

ax = axes[0]
if tri_arr is not None:
    t = mtri.Triangulation(pts[:, 0], pts[:, 1], tri_arr)
    ax.triplot(t, color='lightgray', linewidth=0.2, alpha=0.6)

colour_map = {
    "vertical  (x = cx)":          ('red',    'vertical crease'),
    "horizontal (y = cy)":          ('blue',   'horizontal crease'),
    "diag +45° (y−cy = x−cx)":     ('green',  'diag +45°'),
    "diag −45° (y−cy = −(x−cx))":  ('orange', 'diag −45°'),
}
for name, mask in crease_masks.items():
    c, label = colour_map[name]
    ax.scatter(pts[mask, 0], pts[mask, 1], c=c, s=6, zorder=5, label=f'{label} ({mask.sum()})')

ax.scatter(pts[centre_mask, 0], pts[centre_mask, 1],
           c='purple', s=20, marker='*', zorder=6, label=f'centre pin ({centre_mask.sum()})')
ax.scatter(pts[corner_mask, 0], pts[corner_mask, 1],
           c='black',  s=30, marker='s', zorder=6, label=f'corners UZ ({corner_mask.sum()})')

ax.set_aspect('equal')
ax.legend(fontsize=7, loc='upper right', framealpha=0.9)
ax.set_title(f'Waterbomb mesh — BC regions\n(tol={tol} mm, {len(pts):,} nodes total)')
ax.set_xlabel('X (mm)'); ax.set_ylabel('Y (mm)')

# Z-profile along the vertical crease — shows the V-groove trough geometry
ax2 = axes[1]
vert_mask = crease_masks["vertical  (x = cx)"]
if vert_mask.sum() > 0:
    vp = pts[vert_mask]
    vp = vp[np.argsort(vp[:, 1])]
    ax2.plot(vp[:, 1], vp[:, 2], 'r.-', lw=1.5, label=f'vertical crease  (Z_min={vp[:,2].min():.2f} mm)')

horiz_mask = crease_masks["horizontal (y = cy)"]
if horiz_mask.sum() > 0:
    hp = pts[horiz_mask]
    hp = hp[np.argsort(hp[:, 0])]
    ax2.plot(hp[:, 0], hp[:, 2], 'b.-', lw=1.5, label=f'horizontal crease (Z_min={hp[:,2].min():.2f} mm)')

ax2.axhline(0, color='gray', linestyle=':', lw=0.8, label='flat (Z=0)')
ax2.set_xlabel('Position along crease (mm)'); ax2.set_ylabel('Z (mm)')
ax2.set_title('Z-profile along creases\n(V-groove trough from nTop Trough Depth parameter)')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey insight: the trough geometry is already in the mesh (Z ≠ 0 at creases).")
print("The MAPDL solver uses this geometric nonlinearity to localise bending at creases.")